In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("retail_customer_churn.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

In [ ]:
def evaluate_model(model, X_test, y_test):

    predictions = model.predict(X_test)

    probabilities = model.predict_proba(X_test)[:, 1]

    return {
        "Accuracy": accuracy_score(
            y_test,
            predictions
        ),

        "Precision": precision_score(
            y_test,
            predictions
        ),

        "Recall": recall_score(
            y_test,
            predictions
        ),

        "F1 Score": f1_score(
            y_test,
            predictions
        ),

        "ROC-AUC": roc_auc_score(
            y_test,
            probabilities
        )
    }

In [ ]:
results = []

for name, model in trained_models.items():

    metrics = evaluate_model(
        model,
        X_test,
        y_test
    )

    metrics["Model"] = name

    results.append(metrics)

results_df = pd.DataFrame(results)

results_df = results_df[
    [
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ]
]

results_df.sort_values(
    "ROC-AUC",
    ascending=False
)

In [ ]:
comparison = results_df.set_index("Model")

comparison.plot(
    kind="bar",
    figsize=(14, 7)
)

plt.title("Performance Comparison of Five Classification Models")
plt.ylabel("Score")
plt.ylim(0, 1)

plt.xticks(rotation=30)

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=results_df.sort_values(
        "F1 Score",
        ascending=False
    ),
    x="F1 Score",
    y="Model"
)

plt.title("F1 Score Comparison")
plt.xlim(0, 1)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=results_df.sort_values(
        "ROC-AUC",
        ascending=False
    ),
    x="ROC-AUC",
    y="Model"
)

plt.title("ROC-AUC Comparison")
plt.xlim(0, 1)

plt.show()

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = []

for name, pipeline in pipelines.items():

    print(f"Cross-validating: {name}")

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=[
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc"
        ],
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1": scores["test_f1"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean()
    })

cv_df = pd.DataFrame(cv_results)

cv_df.sort_values(
    "CV ROC-AUC",
    ascending=False
)

In [ ]:
final_comparison = results_df.merge(
    cv_df,
    on="Model"
)

final_comparison.sort_values(
    "CV ROC-AUC",
    ascending=False
)

In [ ]:
best_model_name = (
    final_comparison
    .sort_values(
        "CV ROC-AUC",
        ascending=False
    )
    .iloc[0]["Model"]
)

best_model = trained_models[
    best_model_name
]

print("Best Model:")
print(best_model_name)

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "No Churn",
            "Churn"
        ]
    )
)

In [ ]:
plt.figure(figsize=(10, 7))

for name, model in trained_models.items():

    probabilities = model.predict_proba(
        X_test
    )[:, 1]

    from sklearn.metrics import roc_curve

    fpr, tpr, _ = roc_curve(
        y_test,
        probabilities
    )

    auc = roc_auc_score(
        y_test,
        probabilities
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{name} (AUC={auc:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curves - Five Classification Models")

plt.legend()

plt.show()